In [0]:
from pyspark.sql import functions as F

gold_q1 = spark.table("workspace.rearc_lakeflow.gold_q1_population_stats")

display(gold_q1)

In [0]:
q1 = gold_q1.first()

assert gold_q1.count() == 1, "Q1 must return exactly one row"
assert q1["total_years"] == 6, "Expected 6 years (2013–2018)"
assert q1["start_year"] == 2013
assert q1["end_year"] == 2018
assert abs(q1["mean_population"] - 322069808) < 1
assert abs(q1["stddev_population"] - 4158441.04) < 1

print("PASS: Q1 Gold validation successful")

In [0]:
gold_q2 = spark.table("workspace.rearc_lakeflow.gold_q2_productivity")

total_rows = gold_q2.count()
distinct_series = gold_q2.select("series_id").distinct().count()
duplicate_series = (
    gold_q2.groupBy("series_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert total_rows == 237, f"Expected 237 rows, found {total_rows}"
assert distinct_series == 237, f"Expected 237 unique series, found {distinct_series}"
assert duplicate_series == 0, f"Found {duplicate_series} duplicate series"

print("PASS: Q2 row count and uniqueness validation successful")

In [0]:
gold_q3 = spark.table("workspace.rearc_lakeflow.gold_q3_bls_population")

total_rows = gold_q3.count()

overlap_years = (
    gold_q3
    .filter(
        (F.col("year").between(2013, 2018)) &
        F.col("population").isNotNull()
    )
    .count()
)

assert total_rows == 39, f"Expected 39 rows, found {total_rows}"
assert overlap_years == 6, f"Expected 6 population matches, found {overlap_years}"

assert (
    gold_q3
    .filter(F.col("annual_value").isNull())
    .count() == 0
), "Unexpected null annual values"

print("PASS: Q3 Gold validation successful")

In [0]:
print("================================")
print("REARC DATA QUEST - TEST SUMMARY")
print("================================")
print("Q1: Population statistics - PASS")
print("Q2: Productivity by series - PASS")
print("Q3: Productivity and population - PASS")
print("================================")
print("ALL GOLD VALIDATIONS PASSED")